In [ ]:
from eegprep import loadset
EEG = loadset('/Users/arno/Python/eegprep/sample_data/eeglab_data_with_ica_tmp.set')


In [ ]:
EEG['chanlocs']

In [ ]:
from eegprep import EEGobj

EEG = EEGobj('../sample_data/eeglab_data.set')
EEG.pop_select(channel=['FC1', 'FP1'])
EEG

In [ ]:
EEG = EEGobj.EEG

In [ ]:
from eegprep import EEGobj

EEG = EEGobj('../sample_data/eeglab_data.set')
EEG.select(channel=['FC1', 'FP1'])
EEG

In [ ]:
from eegprep import EEGobj

EEG = EEGobj('../sample_data/eeglab_data.set')
EEG.select(channel=['FC1', 'FP1'])
EEG

In [ ]:
from eegprep import EEGobj

EEG = EEGobj('../sample_data/eeglab_data.set')
EEG.select(channel=['FC1', 'FP1'])
EEG

In [ ]:
from eegprep import EEGobj

EEG = EEGobj('../sample_data/eeglab_data.set')
EEG.select(channel=['FC1', 'FP1'])
EEG

In [ ]:
from eegprep import EEGobj

EEG = EEGobj('../sample_data/eeglab_data.set')
EEG.select(channel=['FC1', 'FP1'])
EEG

In [ ]:
from eegprep import EEGobj

EEG = EEGobj('../sample_data/eeglab_data.set')
EEG.select(channel=['FC1', 'FP1'])
EEG

In [ ]:

EEG = dict({
    "data": 3,
    "xmin": 0.0,
    "xmax": 753.4900,
    "pnts": 2,
    "srate": 100,
    "nbchan": 4,
    "trials": 1,
    "event": [
        {"type": "stim", "latency": 3.0},
        {"type": "boundary", "latency": 6.0, "duration": 0.0},
        {"type": "stim", "latency": 7.0},
        {"type": "resp", "latency": 12.0},
    ],
})
EEG['data']

In [ ]:
import scipy.io
import numpy as np

a = scipy.io.loadmat('a.mat')['a']

print(a)

scipy.io.savemat('a2.mat', {'a': a })

a = np.array([np.array([0.06]), 
              np.array([0.08]), 
              np.array([75.1]), 
              np.array([1.1])], dtype=object)

scipy.io.savemat('a3.mat', {'a': a})

print(a)


In [ ]:
from eegprep.functions.adminfunc.eeglabcompat import get_eeglab

eeglab = get_eeglab('MAT')
result = eeglab.sqrt(4.0)

In [ ]:
result

In [ ]:
import scipy.io
import numpy as np
EEG['event'] = np.ndarray(EEG['event'], dtype=object)
scipy.io.savemat('tmpset.mat', { 'EEG': EEG })

In [ ]:
from copy import copy
import numpy as np
from scipy.linalg import expm
import os
os.environ["OMP_NUM_THREADS"]     = "1"
os.environ["OPENBLAS_NUM_THREADS"] = "1"
os.environ["MKL_NUM_THREADS"]      = "1"

import numpy as np
from picard.densities import Tanh

def core_picard(X, density=Tanh(), ortho=False, extended=False, m=7,
                max_iter=500, tol=1e-7, lambda_min=0.01, ls_tries=10,
                verbose=False, covariance=None):
    '''Runs the Picard algorithm

    The algorithm is detailed in::

        Pierre Ablin, Jean-Francois Cardoso, and Alexandre Gramfort
        Faster independent component analysis by preconditioning with Hessian
        approximations
        IEEE Transactions on Signal Processing, 2018
        https://arxiv.org/abs/1706.08171

    Parameters
    ----------
    X : array, shape (N, T)
        Matrix containing the signals that have to be unmixed. N is the
        number of signals, T is the number of samples. X has to be centered

    m : int
        Size of L-BFGS's memory. Typical values for m are in the range 3-15

    max_iter : int
        Maximal number of iterations for the algorithm

    precon : 1 or 2
        Chooses which Hessian approximation is used as preconditioner.
        1 -> H1
        2 -> H2
        H2 is more costly to compute but can greatly accelerate convergence
        (See the paper for details).

    tol : float
        tolerance for the stopping criterion. Iterations stop when the norm
        of the gradient gets smaller than tol.

    lambda_min : float
        Constant used to regularize the Hessian approximations. The
        eigenvalues of the approximation that are below lambda_min are
        shifted to lambda_min.

    ls_tries : int
        Number of tries allowed for the backtracking line-search. When that
        number is exceeded, the direction is thrown away and the gradient
        is used instead.

    verbose : boolean
        If true, prints the informations about the algorithm.

    Returns
    -------
    Y : array, shape (N, T)
        The estimated source matrix

    W : array, shape (N, N)
        The estimated unmixing matrix, such that Y = WX.
    '''
    # Init
    N, T = X.shape
    W = np.eye(N)
    Y = X
    s_list = []
    y_list = []
    r_list = []
    signs = np.ones(N)
    current_loss = _loss(Y, W, density, signs, ortho, extended)
    print(f'Python initial simple_loss = {current_loss}')
    requested_tolerance = False
    sign_change = False
    gradient_norm = 1.
    if extended:
        if covariance is None:  # Need this for extended
            covariance = X.dot(X.T) / T
        C = covariance.copy()
    for n in range(max_iter):
        # Compute the score function
        psiY, psidY = density.score_and_der(Y)
        # Compute the relative gradient and the Hessian off-diagonal
        G = np.inner(psiY, Y) / T
        del psiY
        # Compute the squared signals
        Y_square = Y ** 2
        # Compute the kurtosis and update the gradient accordingly
        if extended:
            K = np.mean(psidY, axis=1) * np.diag(C)
            K -= np.diag(G)
            signs = np.sign(K)
            if n > 0:
                sign_change = np.any(signs != old_signs)  # noqa
            old_signs = signs  # noqa
            G *= signs[:, None]
            psidY *= signs[:, None]
            if not ortho:  # Like in extended infomax: change the gradient.
                G += C
                psidY += 1
        # Compute the Hessian off diagonal
        if ortho:
            h_off = np.diag(G).copy()
        else:
            h_off = np.ones(N)
        # Compute the Hessian approximation diagonal and regularize
        if ortho:
            psidY_mean = np.mean(psidY, axis=1)
            diag = psidY_mean[:, None] * np.ones(N)[None, :]
            h = 0.5 * (diag + diag.T - h_off[:, None] - h_off[None, :])
            h[h < lambda_min] = lambda_min
        else:
            h = np.inner(psidY, Y_square) / T
            h = _regularize_hessian(h, h_off, lambda_min)
        del psidY, Y_square
        # Project the gradient if ortho
        if ortho:
            G = (G - G.T) / 2
        else:
            G -= np.eye(N)
        # Stopping criterion
        gradient_norm = np.max(np.abs(G))
        if gradient_norm < tol:
            requested_tolerance = True
            break
        # Update the memory
        if n > 0:
            s_list.append(direction)  # noqa
            y = G - G_old  # noqa
            y_list.append(y)
            r_list.append(1. / (np.sum(direction * y)))  # noqa
            if len(s_list) > m:
                s_list.pop(0)
                y_list.pop(0)
                r_list.pop(0)
        G_old = G  # noqa
        # Flush the memory if there is a sign change.
        if extended and sign_change:
            current_loss = None
            s_list, y_list, r_list = [], [], []
        # Find the L-BFGS direction
        direction = _l_bfgs_direction(G, h, h_off, s_list, y_list, r_list,
                                      ortho)
        # Do a line_search in that direction:
        converged, new_Y, new_W, new_loss, direction =\
            _line_search(Y, W, density, direction, signs, current_loss,
                         ls_tries, verbose, ortho, extended)
        if not converged:
            direction = -G
            s_list, y_list, r_list = [], [], []
            _, new_Y, new_W, new_loss, direction =\
                _line_search(Y, W, density, direction, signs, current_loss,
                             10, False, ortho, extended)
        Y = new_Y
        W = new_W
        if covariance is not None:
            C = W.dot(covariance).dot(W.T)
        current_loss = new_loss
        if verbose:
            print('iteration %d, gradient norm = %.6g, loss = %.6g' %
                  (n + 1, gradient_norm, current_loss))
    infos = dict(converged=requested_tolerance, gradient_norm=gradient_norm,
                 n_iterations=n)
    if extended:
        infos['signs'] = signs
    return Y, W, infos


def _loss_simple(Y, W, density, signs, ortho, extended):
    return np.mean(Y**2)

def _loss(Y, W, density, signs, ortho, extended):
    '''
    Computes the loss function for Y, W
    '''
    if not ortho:
        loss = - np.linalg.slogdet(W)[1]
    else:
        loss = 0.
    for y, s in zip(Y, signs):
        loss += s * np.mean(density.log_lik(y))
        if extended and not ortho:
            loss += 0.5 * np.mean(y ** 2)
    return loss


def _line_search(Y, W, density, direction, signs, current_loss, ls_tries,
                 verbose, ortho, extended):
    '''
    Performs a backtracking line search, starting from Y and W, in the
    direction direction. I
    '''
    N = W.shape[0]
    alpha = 1.
    if current_loss is None:
        current_loss = _loss(Y, W, density, signs, ortho, extended)
    for _ in range(ls_tries):
        if ortho:
            transform = expm(alpha * direction)
        else:
            transform = np.eye(N) + alpha * direction
        Y_new = np.dot(transform, Y)
        W_new = np.dot(transform, W)
        new_loss = _loss(Y_new, W_new, density, signs, ortho, extended)
        if new_loss < current_loss:
            return True, Y_new, W_new, new_loss, alpha * direction
        alpha /= 2.
    else:
        if verbose:
            print('line search failed, falling back to gradient')
        return False, Y_new, W_new, new_loss, alpha * direction


def _l_bfgs_direction(G, h, h_off, s_list, y_list, r_list, ortho):
    q = copy(G)
    a_list = []
    for s, y, r in zip(reversed(s_list), reversed(y_list), reversed(r_list)):
        alpha = r * np.sum(s * q)
        a_list.append(alpha)
        q -= alpha * y
    if ortho:
        z = q / h
        z = (z - z.T) / 2.
    else:
        z = _solve_hessian(h, h_off, q)
    for s, y, r, alpha in zip(s_list, y_list, r_list, reversed(a_list)):
        beta = r * np.sum(y * z)
        z += (alpha - beta) * s
    return -z


def _regularize_hessian(h, h_off, lambda_min):
    discr = np.sqrt((h - h.T) ** 2 + 4. * h_off[:, None] * h_off[None, :])
    eigenvalues = 0.5 * (h + h.T - discr)
    # Regularize
    problematic_locs = eigenvalues < lambda_min
    np.fill_diagonal(problematic_locs, False)
    i_pb, j_pb = np.where(problematic_locs)
    h[i_pb, j_pb] += lambda_min - eigenvalues[i_pb, j_pb]
    return h


def _solve_hessian(h, h_off, G):
    det = h * h.T - h_off[:, None] * h_off[None, :]
    return (h.T * G - h_off[:, None] * G.T) / det


In [ ]:
# Authors: Pierre Ablin <pierre.ablin@inria.fr>
#          Alexandre Gramfort <alexandre.gramfort@inria.fr>
#          Jean-Francois Cardoso <cardoso@iap.fr>
#
# License: BSD (3-clause)
from copy import copy
import numpy as np
from scipy.linalg import expm

from picard.densities import Tanh


def core_picard(X, density=Tanh(), ortho=False, extended=False, m=7,
                max_iter=500, tol=1e-7, lambda_min=0.01, ls_tries=10,
                verbose=False, covariance=None):
    '''Runs the Picard algorithm

    The algorithm is detailed in::

        Pierre Ablin, Jean-Francois Cardoso, and Alexandre Gramfort
        Faster independent component analysis by preconditioning with Hessian
        approximations
        IEEE Transactions on Signal Processing, 2018
        https://arxiv.org/abs/1706.08171

    Parameters
    ----------
    X : array, shape (N, T)
        Matrix containing the signals that have to be unmixed. N is the
        number of signals, T is the number of samples. X has to be centered

    m : int
        Size of L-BFGS's memory. Typical values for m are in the range 3-15

    max_iter : int
        Maximal number of iterations for the algorithm

    precon : 1 or 2
        Chooses which Hessian approximation is used as preconditioner.
        1 -> H1
        2 -> H2
        H2 is more costly to compute but can greatly accelerate convergence
        (See the paper for details).

    tol : float
        tolerance for the stopping criterion. Iterations stop when the norm
        of the gradient gets smaller than tol.

    lambda_min : float
        Constant used to regularize the Hessian approximations. The
        eigenvalues of the approximation that are below lambda_min are
        shifted to lambda_min.

    ls_tries : int
        Number of tries allowed for the backtracking line-search. When that
        number is exceeded, the direction is thrown away and the gradient
        is used instead.

    verbose : boolean
        If true, prints the informations about the algorithm.

    Returns
    -------
    Y : array, shape (N, T)
        The estimated source matrix

    W : array, shape (N, N)
        The estimated unmixing matrix, such that Y = WX.
    '''
    # Init
    N, T = X.shape
    W = np.eye(N)
    Y = X
    s_list = []
    y_list = []
    r_list = []
    signs = np.ones(N)
    current_loss = _loss(Y, W, density, signs, ortho, extended)
    requested_tolerance = False
    sign_change = False
    gradient_norm = 1.
    if extended:
        if covariance is None:  # Need this for extended
            covariance = X.dot(X.T) / T
        C = covariance.copy()
    for n in range(max_iter):
        # Compute the score function
        psiY, psidY = density.score_and_der(Y)
        # Compute the relative gradient and the Hessian off-diagonal
        G = np.inner(psiY, Y) / T
        del psiY
        # Compute the squared signals
        Y_square = Y ** 2
        # Compute the kurtosis and update the gradient accordingly
        if extended:
            K = np.mean(psidY, axis=1) * np.diag(C)
            K -= np.diag(G)
            signs = np.sign(K)
            if n > 0:
                sign_change = np.any(signs != old_signs)  # noqa
            old_signs = signs  # noqa
            G *= signs[:, None]
            psidY *= signs[:, None]
            if not ortho:  # Like in extended infomax: change the gradient.
                G += C
                psidY += 1
        # Compute the Hessian off diagonal
        if ortho:
            h_off = np.diag(G).copy()
        else:
            h_off = np.ones(N)
        # Compute the Hessian approximation diagonal and regularize
        if ortho:
            psidY_mean = np.mean(psidY, axis=1)
            diag = psidY_mean[:, None] * np.ones(N)[None, :]
            h = 0.5 * (diag + diag.T - h_off[:, None] - h_off[None, :])
            h[h < lambda_min] = lambda_min
        else:
            h = np.inner(psidY, Y_square) / T
            h = _regularize_hessian(h, h_off, lambda_min)
        del psidY, Y_square
        # Project the gradient if ortho
        if ortho:
            G = (G - G.T) / 2
        else:
            G -= np.eye(N)
        # Stopping criterion
        gradient_norm = np.max(np.abs(G))
        if gradient_norm < tol:
            requested_tolerance = True
            break
        # Update the memory
        if n > 0:
            s_list.append(direction)  # noqa
            y = G - G_old  # noqa
            y_list.append(y)
            r_list.append(1. / (np.sum(direction * y)))  # noqa
            if len(s_list) > m:
                s_list.pop(0)
                y_list.pop(0)
                r_list.pop(0)
        G_old = G  # noqa
        # Flush the memory if there is a sign change.
        if extended and sign_change:
            current_loss = None
            s_list, y_list, r_list = [], [], []
        # Find the L-BFGS direction
        direction = _l_bfgs_direction(G, h, h_off, s_list, y_list, r_list,
                                      ortho)
        # Do a line_search in that direction:
        converged, new_Y, new_W, new_loss, direction =\
            _line_search(Y, W, density, direction, signs, current_loss,
                         ls_tries, verbose, ortho, extended)
        if not converged:
            direction = -G
            s_list, y_list, r_list = [], [], []
            _, new_Y, new_W, new_loss, direction =\
                _line_search(Y, W, density, direction, signs, current_loss,
                             10, False, ortho, extended)
        Y = new_Y
        W = new_W
        if covariance is not None:
            C = W.dot(covariance).dot(W.T)
        current_loss = new_loss
        if verbose:
            print('iteration %d, gradient norm = %.6g, loss = %.6g' %
                  (n + 1, gradient_norm, current_loss))
    infos = dict(converged=requested_tolerance, gradient_norm=gradient_norm,
                 n_iterations=n)
    if extended:
        infos['signs'] = signs
    return Y, W, infos


def _loss(Y, W, density, signs, ortho, extended):
    '''
    Computes the loss function for Y, W
    '''
    if not ortho:
        loss = - np.linalg.slogdet(W)[1]
    else:
        loss = 0.
    for y, s in zip(Y, signs):
        loss += s * np.mean(density.log_lik(y))
        if extended and not ortho:
            loss += 0.5 * np.mean(y ** 2)
    return loss


def _line_search(Y, W, density, direction, signs, current_loss, ls_tries,
                 verbose, ortho, extended):
    '''
    Performs a backtracking line search, starting from Y and W, in the
    direction direction. I
    '''
    N = W.shape[0]
    alpha = 1.
    if current_loss is None:
        current_loss = _loss(Y, W, density, signs, ortho, extended)
    for _ in range(ls_tries):
        if ortho:
            transform = expm(alpha * direction)
        else:
            transform = np.eye(N) + alpha * direction
        Y_new = np.dot(transform, Y)
        W_new = np.dot(transform, W)
        new_loss = _loss(Y_new, W_new, density, signs, ortho, extended)
        if new_loss < current_loss:
            return True, Y_new, W_new, new_loss, alpha * direction
        alpha /= 2.
    else:
        if verbose:
            print('line search failed, falling back to gradient')
        return False, Y_new, W_new, new_loss, alpha * direction


def _l_bfgs_direction(G, h, h_off, s_list, y_list, r_list, ortho):
    q = copy(G)
    a_list = []
    for s, y, r in zip(reversed(s_list), reversed(y_list), reversed(r_list)):
        alpha = r * np.sum(s * q)
        a_list.append(alpha)
        q -= alpha * y
    if ortho:
        z = q / h
        z = (z - z.T) / 2.
    else:
        z = _solve_hessian(h, h_off, q)
    for s, y, r, alpha in zip(s_list, y_list, r_list, reversed(a_list)):
        beta = r * np.sum(y * z)
        z += (alpha - beta) * s
    return -z


def _regularize_hessian(h, h_off, lambda_min):
    discr = np.sqrt((h - h.T) ** 2 + 4. * h_off[:, None] * h_off[None, :])
    eigenvalues = 0.5 * (h + h.T - discr)
    # Regularize
    problematic_locs = eigenvalues < lambda_min
    np.fill_diagonal(problematic_locs, False)
    i_pb, j_pb = np.where(problematic_locs)
    h[i_pb, j_pb] += lambda_min - eigenvalues[i_pb, j_pb]
    return h


def _solve_hessian(h, h_off, G):
    det = h * h.T - h_off[:, None] * h_off[None, :]
    return (h.T * G - h_off[:, None] * G.T) / det


In [ ]:
import numpy as np
from scipy.io import savemat, loadmat
from picard import _core_picard
    
def amari_index(W_true, W_est):
    P = W_est @ np.linalg.inv(W_true)
    C = np.abs(P)
    row_sum = np.sum(C, axis=1, keepdims=True)
    col_sum = np.sum(C, axis=0, keepdims=True)
    r = np.sum((C / row_sum - 1/P.shape[1])**2)
    c = np.sum((C / col_sum - 1/P.shape[0])**2)
    return (r + c) / (2 * P.shape[0])

# fixed seed for reproducibility
np.random.seed(0)

# generate sources
N, T = 5, 10000
S = np.random.laplace(size=(N, T))

# random mixing
A = np.random.randn(N, N)
X = A @ S

# save data for MATLAB
savemat('picard_data.mat', {'X': X, 'A': A})

# run Python Picard
# from threadpoolctl import threadpool_limits
#with threadpool_limits(limits=1, user_api="blas"):
Y_py, W_py, info_py = core_picard(X.copy(), ortho=False, extended=False,
                                  max_iter=200, tol=1e-6, m=10,
                                  lambda_min=0.01, ls_tries=10, verbose=True)
print('Python converged in', info_py['n_iterations'], 'iterations')


In [ ]:
np.dot(W_py, Y_py).shape

In [ ]:
X.dtype

In [ ]:
from picard.densities import Tanh, check_density
check_density(Tanh(), tol=1e-6)

In [ ]:
# compare the pArray and mArray by plotting them
import matplotlib
import matplotlib.pyplot as plt
import numpy as np

# Plotting for visual comparison
fig, axes = plt.subplots(1, 3, figsize=(18, 5))

im1 = axes[0].imshow(pArray, aspect='auto', cmap='viridis')
axes[0].set_title('Python icaweights')
fig.colorbar(im1, ax=axes[0])

im2 = axes[1].imshow(mArray, aspect='auto', cmap='viridis')
axes[1].set_title('MATLAB icaweights')
fig.colorbar(im2, ax=axes[1])

diff = np.abs(pArray - mArray)
im3 = axes[2].imshow(diff, aspect='auto', cmap='magma')
axes[2].set_title('Absolute Difference')
fig.colorbar(im3, ax=axes[2])

plt.suptitle('Comparison of ICA Weight Matrices')
plt.tight_layout(rect=[0, 0.03, 1, 0.95])
plt.show()


In [ ]:
import mne
raw = mne.io.read_raw_eeglab('./sub-NDARDB033FW5_task-RestingState_eeg.set', preload=True)
raw

# Print events from the raw object
events = mne.events_from_annotations(raw)
print("\nEvents found:")
print(events)

# Print annotations
print("\nAnnotations:")
print(raw.annotations)

# You can also get more detailed information about the annotations
print("\nAnnotation details:")
for ann in raw.annotations:
    print(f"Onset: {ann['onset']:.2f}s, Duration: {ann['duration']:.2f}s, Description: {ann['description']}")

In [ ]:
# Extract segments after "instructed_toCloseEyes" events
import numpy as np

# Get all "instructed_toCloseEyes" events from raw annotations
close_eyes_events = []
for annot in raw.annotations:
    if annot['description'] == 'instructed_toCloseEyes':
        close_eyes_events.append(annot['onset'])

# Define segment parameters
post_event_delay = 5  # seconds after event
sfreq = raw.info['sfreq']  # sampling frequency

# Extract segments
segments = []
for event_time in close_eyes_events:
    for event_add_time in range(0, 34, 2): # every 2 seconds for 34 seconds
        start_sample = int((event_time + post_event_delay + event_add_time) * sfreq)
        end_sample = start_sample + int(2 * sfreq)
    
        # Check if segment extends beyond data
        if end_sample <= raw.n_times:
            segment = raw.get_data(start=start_sample, stop=end_sample)
            segments.append(segment)

segments = np.array(segments)  # Shape will be (n_segments, n_channels, n_samples)

print(f"Found {len(close_eyes_events)} 'instructed_toCloseEyes' events")
print(f"Extracted {len(segments)} segments of {segment_duration} seconds each")
print(f"Segment shape: {segments.shape}")

In [ ]:
# Extract 2-second segments after "instructed_toCloseEyes" events
import mne
raw = mne.io.read_raw_eeglab('./sub-NDARDB033FW5_task-RestingState_eeg.set', preload=True)

# Create events array from annotations
events, event_id = mne.events_from_annotations(raw)

# Create new events array for 2-second segments
new_events = []
sfreq = raw.info['sfreq']
for event in events[events[:, 2] == event_id['instructed_toCloseEyes']]:
    # For each original event, create events every 2 seconds from 5s to 39s after
    start_times = event[0] + np.arange(15, 29, 2) * sfreq
    new_events.extend([[int(t), 0, 1] for t in start_times])

for event in events[events[:, 2] == event_id['instructed_toOpenEyes']]:
    # For each original event, create events every 2 seconds from 5s to 39s after
    start_times = event[0] + np.arange(5, 19, 2) * sfreq
    new_events.extend([[int(t), 0, 2] for t in start_times])

# replace events in raw
new_events = np.array(new_events)
annot_from_events = mne.annotations_from_events(
    events=new_events,
    event_desc={1: 'eyes_closed', 2: 'eyes_open'},
    sfreq=raw.info['sfreq']
)

# Clear existing annotations and set new ones
raw.set_annotations(annot_from_events)
# Create Epochs object with 2-second segments
epochs = mne.Epochs(raw,
                   event_id={'eyes_closed': 1, 'eyes_open': 2},
                   tmin=0,
                   tmax=1.998,
                   baseline=None,
                   preload=True)

print(f"Number of 2-second segments: {len(epochs)}")
print(f"Data shape: {epochs.get_data().shape}")


In [ ]:
from braindecode.preprocessing import Preprocessor
import numpy as np
import mne

class hbn_ec_ec_reanotation(Preprocessor):
    def __init__(self):
        pass    
    
    def apply(self, raw):
        # Create events array from annotations
        events, event_id = mne.events_from_annotations(raw)

        # Create new events array for 2-second segments
        new_events = []
        sfreq = raw.info['sfreq']
        for event in events[events[:, 2] == event_id['instructed_toCloseEyes']]:
            # For each original event, create events every 2 seconds from 15s to 29s after
            start_times = event[0] + np.arange(15, 29, 2) * sfreq
            new_events.extend([[int(t), 0, 1] for t in start_times])

        for event in events[events[:, 2] == event_id['instructed_toOpenEyes']]:
            # For each original event, create events every 2 seconds from 5s to 19s after
            start_times = event[0] + np.arange(5, 19, 2) * sfreq
            new_events.extend([[int(t), 0, 2] for t in start_times])

        # replace events in raw
        new_events = np.array(new_events)
        annot_from_events = mne.annotations_from_events(events=new_events,event_desc={1: 'eyes_closed', 2: 'eyes_open'},sfreq=raw.info['sfreq'])
        raw.set_annotations(annot_from_events)        
        return raw

# Example usage:
processor = hbn_ec_ec_reanotation()
raw = mne.io.read_raw_eeglab('./sub-NDARDB033FW5_task-RestingState_eeg.set', preload=True)
raw2 = processor.apply(raw)

events = mne.events_from_annotations(raw)
print("\nEvents found:")
print(events)


In [ ]:
new_events


In [ ]:
a = range(0,10, 2)
a[3]

In [ ]:
# USE PYTHON ASR
import mne

sample_data_path = mne.datasets.sample.data_path()
eeg_file = str(sample_data_path) + '/MEG/sample/sample_audvis_raw.fif'

# Read the raw EEG data
raw = mne.io.read_raw_fif(eeg_file, preload=True)
print(raw.info)

import asrpy
asr = asrpy.ASR(sfreq=raw.info["sfreq"], cutoff=20)
asr.fit(raw)
raw = asr.transform(raw)

In [ ]:
import sys
import importlib
sys.path.insert(0, '/Users/arno/Python/eegprep/src/')
import eegprep
importlib.reload(eegprep)  # Step 3: Reload the library to reflect the changes
from eegprep import pop_loadset
from eegprep import iclabel

# Load the dataset
eeg_file = '../sample_data/eeglab_data_with_ica_tmp.set'
EEG = pop_loadset(eeg_file)
# EEG = pop_eegfiltnew(EEG, locutoff=5,hicutoff=25,revfilt=True,plotfreqz=False)
# EEG = clean_artifacts(EEG, FlatlineCriterion=5,ChannelCriterion=0.87, LineNoiseCriterion=4,Highpass=[0.25, 0.75],BurstCriterion= 20, WindowCriterion=0.25, BurstRejection=True, WindowCriterionTolerances=[float('-inf'), 7])
EEG = iclabel(EEG)
# pop_saveset(EEG, fname_out)

In [ ]:
from oct2py import octave as eeglab
sys.path.insert(0, '/Users/arno/Python/eegprep/src/')
import eegprep
importlib.reload(eegprep)  # Step 3: Reload the library to reflect the changes
from eegprep import pop_loadset
from eegprep.functions.adminfunc import eeglabcompat
import time

path2eeglab = '/System/Volumes/Data/data/matlab/eeglab/'
eeglab.addpath(path2eeglab + '/functions/guifunc')
eeglab.addpath(path2eeglab + '/functions/popfunc')
eeglab.addpath(path2eeglab + '/functions/adminfunc')
eeglab.addpath(path2eeglab + '/plugins/firfilt')
eeglab.addpath(path2eeglab + '/functions/sigprocfunc')
eeglab.addpath(path2eeglab + '/functions/miscfunc')
eeglab.addpath(path2eeglab + '/plugins/dipfit')

start = time.time()
EEG = eeglab.pop_loadset(path2eeglab + '/sample_data/eeglab_data_epochs_ica.set')
EEG = eeglab.pop_loadset('/Users/arno/Python/eegprep/tmp.set')
stop = time.time()
print(f"Time taken pop_loadset: {stop - start}")

# time the function below
import time
start = time.time()
EEG2 = eeglab.pop_resample(EEG, 100)
stop = time.time()
print(f"Time taken pop_resample: {stop - start}")


In [ ]:
import eegprep
importlib.reload(eegprep)  # Step 3: Reload the library to reflect the changes
from eegprep import pop_loadset
from eegprep import pop_eegfiltnew
from eegprep import clean_artifacts

eeglab_file_path = '/System/Volumes/Data/data/matlab/eeglab/sample_data/eeglab_data.set'
EEG = pop_loadset(eeglab_file_path)
EEG = pop_eegfiltnew(EEG, locutoff=5,hicutoff=25,revfilt=True,plotfreqz=False)
EEG = clean_artifacts(EEG, FlatlineCriterion=5,ChannelCriterion=0.87, LineNoiseCriterion=4,Highpass=False,BurstCriterion= 20, WindowCriterion=0.25, BurstRejection=False, WindowCriterionTolerances=[float('-inf'), 7])


In [ ]:
import pop_loadset
import importlib
import matplotlib.pyplot as plt
importlib.reload(pop_loadset)  # Step 3: Reload the library to reflect the changes
from pop_loadset import pop_loadset

eeglab_file_path = '/System/Volumes/Data/data/matlab/eeglab/sample_data/eeglab_data_epochs_ica.set'
#eeglab_file_path = 'NDARZZ830JM7_eyesclosed.set'
EEG2 = pop_loadset(eeglab_file_path)

print(EEG2['data'][0,2,1])

# eeglab.eeg_compare(EEG, EEG2)

# print(EEG['filepath'])
# print(EEG2['filepath'])

# Field filepath differs
# Field icaact differs
# Field icawinv differs
# Field icaweights differs
# Field chanlocs differs
# Field urchanlocs differs
# Field chaninfo differs
# Field event differs (n=157 vs n=157)
# Field urevent differs
# Field epoch differs
# Field epochdescription differs
# Field reject differs
# Field history differs
# Field saved differs
# Field etc differs
# Field run missing in second dataset
# Field roi missing in second dataset
# The field above are different between the two datasets
# print them all
# print(EEG.keys())
# print(EEG2.keys())
# for key in EEG.keys():
#     if key in EEG2.keys():
#         if not np.all(EEG[key] == EEG2[key]):
#             print(key)
#     else:
#         print(key)
# for key in EEG2.keys():
#     if key not in EEG.keys():
#         print(key)

# check if EEG exists in the workspace
if 'EEG' in locals():
    print(EEG['filepath'])
    print(EEG2['filepath'])
    print(EEG['icaact'].shape)
    print(EEG2['icaact'].shape)
    print(EEG['icawinv'].shape)
    print(EEG2['icawinv'].shape)
    print(EEG['icaweights'].shape)
    print(EEG2['icaweights'].shape)
    print(EEG['chanlocs'].shape)
    print(EEG2['chanlocs'].shape)
    print(EEG['urchanlocs'].shape)
    print(EEG2['urchanlocs'].shape)

    EEG['chanlocs']['labels'][0,0]

    EEG.event.type[0,0]
    EEG['event']['type'][0,0]
    EEG2['event'][0]['type']

In [ ]:
a = np.array([1, 2, 3])

# check if a is a numpy array
if isinstance(a, np.ndarray):
    print('a is a numpy array')

In [ ]:
import numpy as np

if len(EEG['icachansind']) == EEG['nbchan']:
    # Subtract mean from EEG data using broadcasting in NumPy
    EEG['data'] = EEG['data'] - np.mean(EEG['data'], axis=0)
    
    # Subtract mean from EEG icawinv using broadcasting in NumPy
    EEG['icawinv'] = EEG['icawinv'] - np.mean(EEG['icawinv'], axis=0)
    
    # Compute the pseudoinverse of EEG['icawinv']
    EEG['icaweights'] = np.linalg.pinv(EEG['icawinv'])
    
    # Create an identity matrix for EEG['icasphere']
    EEG['icasphere'] = np.eye(EEG['nbchan'])
    
    # Set EEG['ref'] to 'average'
    EEG['ref'] = 'average'
    
    # Update the reference for each channel in EEG['chanlocs']
    for iChan in range(len(EEG['chanlocs'])):
        EEG['chanlocs'][iChan]['ref'] = 'average'

In [ ]:
# Create the list of dictionaries
d_list = [{'labels': c} for c in ['a','b','c']]

# Define the data type for the structured array
dtype = np.dtype([('labels', np.str_)])

# Convert the list of dictionaries to a structured NumPy array
tmp = np.array([ (item['labels'],) for item in d_list ], dtype=dtype)



In [ ]:
# save_d_to_mat.py

import numpy as np
from scipy.io import savemat

# Create the list of dictionaries
d_list = [{'labels': c} for c in range(2)]

# Define the data type for the structured array
dtype = np.dtype([('labels', np.int32)])

# Convert the list of dictionaries to a structured NumPy array
d_array = np.array([ (item['labels'],) for item in d_list ], dtype=dtype)

# Save the structured array to a MATLAB .mat file
savemat('d_struct2.mat', {'d': d_array})

In [ ]:
from topoplot import topoplot
from pop_loadset import pop_loadset
from pop_saveset import pop_saveset
import os
import time

eeglab_file_path = '/System/Volumes/Data/data/matlab/eeglab/sample_data/eeglab_data_epochs_ica.set'
eeglab_file_path = './eeglab_data_with_ica_tmp.set'
start = time.time()
EEG = pop_loadset(eeglab_file_path)
stop = time.time()
print(f"Time taken pop_loadset(p): {stop - start}")

start = time.time()
EEG = pop_saveset(EEG, 'tmp.set')
stop = time.time()
print(f"Time taken pop_saveset(p): {stop - start}")

# plot data using MNE

import mne
import matplotlib.pyplot as plt

# create MNE info structure
info = mne.create_info(ch_names=[ x['labels'] for x in EEG['chanlocs']], sfreq=EEG['srate'], ch_types='eeg')

# create MNE Raw object
raw = mne.io.RawArray(EEG['data'].transpose(0,2,1).reshape(EEG['nbchan'], -1), info)

# plot data
raw.plot(n_channels=10, scalings=100, title='Data from arrays', show=True, block=True, duration=10.0)    

# plot topoplot
# topoplot(EEG['icawinv'][:, 0], EEG['chanlocs'], show=True)

In [ ]:
import pop_loadset
import importlib
import matplotlib.pyplot as plt
importlib.reload(pop_loadset)  # Step 3: Reload the library to reflect the changes
from pop_loadset import pop_loadset

EEG = pop_loadset(eeglab_file_path)
EEG['data']

nfreqs = 100
pct_data = 100

# clean input cutoff freq
nyquist = EEG['srate'] // 2
if nfreqs is None or nfreqs > nyquist:
    nfreqs = nyquist

# setup constants
ncomp = EEG['icaweights'].shape[0]

# Hamming window
n_points = min(EEG['pnts'], EEG['srate'])
m = n_points
isOddLength = m % 2
if isOddLength:
    x = np.arange(0, (m - 1) / 2 + 1) / (m - 1)
else:
    x = np.arange(0, m / 2) / (m - 1)

a = 0.54
window = a - (1 - a) * np.cos(2 * np.pi * x)
if isOddLength:
    window = np.concatenate([window, window[-2::-1]])
else:
    window = np.concatenate([window, window[::-1]])

cutoff = (EEG['pnts'] // n_points) * n_points
index = np.add.outer(np.arange(0, cutoff - n_points//2, n_points // 2), np.arange(0, n_points)).astype(int).transpose()

np.random.seed(42)  # rng('default') in MATLAB
n_seg = index.shape[1] * EEG['trials']
num_vals = int(n_seg * pct_data / 100)
subset = np.arange(0, num_vals)
random_numbers = np.random.rand(num_vals)
sort_idx = np.argsort(random_numbers)
subset = subset[sort_idx]

# subset = np.random.permutation(n_seg)[:int(n_seg * pct_data / 100)]

# calculate windowed spectrums
psdmed = np.zeros((ncomp, nfreqs))
for it in range(ncomp):
    temp = np.reshape(EEG['icaact'][it, index, :], (1, index.shape[0], index.shape[1] * EEG['trials']))
    temp = temp[:, :, subset] * window[:, np.newaxis]
    temp = fft(temp, n_points, axis=1)
    temp = np.abs(temp) ** 2
    temp = temp[:, 1:nfreqs + 1, :] * 2 / (EEG['srate'] * np.sum(window ** 2))
    if nfreqs == nyquist:
        temp[:, -1, :] /= 2
    psdmed[it, :] = 20 * np.log10(np.median(temp, axis=2))

# image psdmed
plt.imshow(psdmed, aspect='auto', origin='lower', extent=[0, nfreqs, 0, ncomp])
plt.colorbar()
plt.show()

In [ ]:
import pop_loadset
importlib.reload(pop_loadset)  # Step 3: Reload the library to reflect the changes
from pop_loadset import pop_loadset
eeglab_file_path = '/System/Volumes/Data/data/matlab/eeglab/sample_data/eeglab_data_epochs_ica.set'
EEG = pop_loadset(eeglab_file_path)
# EEG['data'].transpose(2, 0, 1).shape
# EEG['data'][:2,:2,:2]
EEG['data'][5,10,11]
# EEG['data'][:,1,0]

In [ ]:
plt.imshow(temp[:,:].squeeze(), aspect='auto', origin='lower', interpolation='none')
plt.colorbar()
plt.show()


In [ ]:
import numpy as np

# Set the seed for reproducibility
np.random.seed(42)

# Generate random numbers and use them to shuffle the array
array = np.arange(1, 11)  # Create an array [1, 2, ..., 10]
random_numbers = np.random.rand(len(array))
sort_idx = np.argsort(random_numbers)
shuffled_array = array[sort_idx]

print(shuffled_array)

In [ ]:
temp.squeeze().shape

In [ ]:
import pop_loadset
import importlib
importlib.reload(pop_loadset)  # Step 3: Reload the library to reflect the changes
from pop_loadset import pop_loadset

var = pop_loadset('var.mat')
var['test'][2]

In [ ]:
import pop_loadset
import importlib
importlib.reload(pop_loadset)  # Step 3: Reload the library to reflect the changes
from pop_loadset import pop_loadset
import numpy as np
from scipy.signal import resample
from numpy.fft import fft, ifft
import matplotlib.pyplot as plt



eeglab_file_path = './eeglab_data_with_ica_tmp.set'
EEG = pop_loadset(eeglab_file_path)


# convert EEG['event'] to list
EEG['event'] = EEG['event'].tolist()

# save EEG dictionary as a .mat file
import scipy.io
scipy.io.savemat('EEG2.mat', EEG)

EEG['event'][0]

# create np.dtype for the structured array EEG['event'] in a general way scanning all event fields
dtype = np.dtype([(key, 'O') for key in EEG['event'][0].keys()])





In [ ]:
from eeg_mne2eeglab import eeg_mne2eeglab

np.arange(0, 10, 2)

In [ ]:
import numpy as np
from scipy.signal import resample
#from numpy.fft import fft, ifft
from scipy.fft import fft, ifft
import random
import pop_loadset
import importlib
importlib.reload(pop_loadset)  # Step 3: Reload the library to reflect the changes
from pop_loadset import pop_loadset
import numpy as np
from scipy.signal import resample
from numpy.fft import fft, ifft
import matplotlib.pyplot as plt
import scipy.io

eeglab_file_path = './eeglab_data_with_ica_tmp.set'
#eeglab_file_path = '/System/Volumes/Data/data/matlab/eeglab/sample_data/eeglab_data_epochs_ica.set'
EEG = pop_loadset(eeglab_file_path)
#isinstance(EEG['chanlocs'][0], scipy.io.matlab._mio5_params.mat_struct)

# EEG = {
#     'srate': 256,
#     'icaweights': np.random.randn(10, 256),
#     'pnts': 1000,
#     'trials': 5,
#     'icaact': np.random.randn(10, 1000, 5)
# }
    
pct_data = None

#def eeg_autocorr_welch(EEG, pct_data=100):
if pct_data is None or pct_data == 0:
    # clean input cutoff freq
    if pct_data is None or pct_data == 0:
        pct_data = 100
    
    # setup constants
    ncomp = EEG['icaweights'].shape[0]
    n_points = min(EEG['pnts'], EEG['srate'] * 3)
    nfft = 2**(int(np.log2(n_points * 2 - 1)) + 1)
    cutoff = (EEG['pnts'] // n_points) * n_points
    index = np.add.outer(np.ceil(np.arange(0, cutoff - n_points, n_points // 2)).astype(int), np.arange(n_points)).astype(int)
    
    # separate data segments
    if pct_data != 100:
        random.seed(0)
        n_seg = index.shape[0] * EEG['trials']
        subset = random.sample(range(n_seg), int(np.ceil(n_seg * pct_data / 100)))
        random.seed()  # restore normal random behavior
        temp = np.reshape(EEG['icaact'][:, index, :], (ncomp, *index.shape, EEG['trials']))
        segments = temp[:, :, subset]
    else:
        segments = np.reshape(EEG['icaact'][:, index, :], (ncomp, *index.shape, EEG['trials']))
    
    # calc autocorrelation
    ac = np.zeros((ncomp, nfft))
    for it in range(ncomp):
        fftpow = np.mean(np.abs(fft(segments[it, :, :], nfft, axis=1))**2, axis=2)
        ac[it, :] = np.real(ifft(fftpow, axis=0))
    
    # normalize
    if EEG['pnts'] < EEG['srate']:
        ac = np.concatenate([ac[:, :EEG['pnts']] / (ac[:, 0][:, np.newaxis] * np.arange(n_points, 0, -1) / n_points), 
                             np.zeros((ncomp, EEG['srate'] - n_points + 1))], axis=1)
    else:
        ac = ac[:, :EEG['srate'] + 1] / (ac[:, 0][:, np.newaxis] * np.concatenate([np.arange(n_points, n_points - EEG['srate'], -1), 
                                                                                   [max(1, n_points - EEG['srate'])]] / n_points))
    
    # resample to 1 second at 100 samples/sec
    ac = resample(ac.T, 100, axis=0).T
    ac = ac[:, 1:101]
   
    # resample to 1 second at 100 samples/sec
    ac = resample(ac.T, 100, axis=0).T
    ac = ac[:, 1:101]
    
    #return ac

# Assuming EEG class or data structure is defined with appropriate fields
# Example usage:
# ac = eeg_autocorr_welch(EEG, pct_data=50)
    # print information about psdmed
   
    
    #assert psdmed.shape == (10, 100)
    #assert np.all(np.isfinite(psdmed))

#test_eeg_autocorr()

In [ ]:
import numpy as np
from scipy.signal import resample_poly

# Generate random numbers between 69 and 251
ac = np.random.randint(69, 251, size=(251,))  # 251 samples as in MATLAB

# Resample to match MATLAB's behavior
resamp = resample_poly(ac, up=100, down=250)

# Get the size of the resampled array
size_resamp = resamp.shape[0]

print(size_resamp)  # Should be 101

In [ ]:
import pop_loadset
import importlib
importlib.reload(pop_loadset)  # Step 3: Reload the library to reflect the changes
from pop_loadset import pop_loadset
import numpy as np
from scipy.signal import resample
from numpy.fft import fft, ifft
import matplotlib.pyplot as plt
import scipy.io

eeglab_file_path = './eeglab_data_with_ica_tmp.set'
#eeglab_file_path = '/System/Volumes/Data/data/matlab/eeglab/sample_data/eeglab_data_epochs_ica.set'
EEG = pop_loadset(eeglab_file_path)
#isinstance(EEG['chanlocs'][0], scipy.io.matlab._mio5_params.mat_struct)

EEG



In [ ]:
import numpy as np

cutoff = 375
n_points = 125
index = np.add.outer(np.arange(0, cutoff - n_points//2 + 1, n_points // 2), np.arange(0, n_points)).astype(int).transpose()
index
np.ceil(np.arange(0, cutoff - n_points + 1, n_points/2)).astype(int)
index = np.add.outer(np.ceil(np.arange(0, cutoff - n_points + 1, n_points/2)).astype(int), np.arange(0, n_points)).astype(int).transpose()
index



In [ ]:
data = scipy.io.loadmat('python_temp.mat')
data['grid'][0][2]

ic_classes = EEG['etc']['ic_classification']\
    ['ICLabel']['classifications']

In [ ]:
from pop_loadset import pop_loadset

# Create an in-memory bytes buffer
eeglab_file_path = './tmp2.set'
#eeglab_file_path = '/System/Volumes/Data/data/matlab/eeglab/sample_data/eeglab_data_epochs_ica.set'
EEG = pop_loadset(eeglab_file_path)

In [ ]:
import numpy as np

# Example initialization for features (mock data)
features = [
    np.random.rand(5, 5, 5, 5).astype(np.float32),  # Random 4D array
    np.random.rand(5, 5, 5, 1).astype(np.float32),  # Random 4D array
    np.random.rand(5, 5, 5, 1).astype(np.float32)   # Random 4D array
]

# Check initial shapes
print("Initial shapes:")
for i, feature in enumerate(features):
    print(f"features[{i}]:", feature.shape)

# Equivalent of MATLAB code for features[0]
features[0] = np.single(
    np.concatenate([
        features[0],                                   # Original array
        -features[0],                                  # Negated array
        features[0][:, ::-1, :, :],                    # Array with 2nd dimension reversed
        -features[0][:, ::-1, :, :]                    # Negated reversed array
    ], axis=3)                                         # Concatenate along the 4th dimension
)
print("features[0] shape after concatenation:", features[0].shape)

# Equivalent of MATLAB code for features[1]
features[1] = np.single(
    np.tile(features[1], (1, 1, 1, 4))                 # Replicate along 4th dimension
)
print("features[1] shape after tiling:", features[1].shape)

# Equivalent of MATLAB code for features[2]
features[2] = np.single(
    np.tile(features[2], (1, 1, 1, 4))                 # Replicate along 4th dimension
)
print("features[2] shape after tiling:", features[2].shape)

# Verify data types
for i, feature in enumerate(features):
    print(f"features[{i}] dtype:", feature.dtype)